# 02 · Data Preprocessing

**Input:** `data/raw/climate_monthly_raw.csv` (Earth Engine export, see `gee/extract_data.py`) and `data/raw/districts_static.csv`
**Output:** `data/processed/climate_monthly.parquet`: one clean row per district per month.

Steps:
1. Load and merge the climate and static district data
2. Data-quality checks (missing values, duplicates, incomplete months, implausible values)
3. Missing-value strategy
4. Unit verification
5. Time columns and seasons
6. Climatology and anomalies (leakage-safe)
7. A note on normalisation / standardisation

All the logic lives in `src/preprocessing.py`, so the EDA notebook, model notebooks and dashboard use exactly the same processing.

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd, numpy as np
from src import config as C, preprocessing as P
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

## 1. Load and merge

In [2]:
climate, static = P.load_raw()
print("climate:", climate.shape, "| static:", static.shape)
df = climate.merge(static, on="district_id", how="left", validate="many_to_one")
df.head()

climate: (39312, 12) | static: (126, 7)


,district_id,year,month,tmax_c,tmin_c,tmean_c,soil_moisture,solar_mj,era5_rain_mm,rain_mm,era5_days,chirps_days,district,province,lat,lon,area_km2,elevation_m
0,60131773B12131197496886,2000,1,18.332232,6.420995,12.239377,0.217611,10.328772,56.855826,27.306307,31,31,Lahore,Punjab,31.433204,74.341424,1694.990734,210.646812
1,60131773B12131197496886,2000,2,18.774428,6.407301,12.657003,0.269148,12.533770,38.704094,3.295176,29,29,Lahore,Punjab,31.433204,74.341424,1694.990734,210.646812
2,60131773B12131197496886,2000,3,27.123301,11.624535,19.546631,0.145434,17.120793,9.467841,3.331117,31,31,Lahore,Punjab,31.433204,74.341424,1694.990734,210.646812
3,60131773B12131197496886,2000,4,37.186564,19.843533,28.933908,0.130942,20.083929,4.695242,7.884926,30,30,Lahore,Punjab,31.433204,74.341424,1694.990734,210.646812
4,60131773B12131197496886,2000,5,41.067840,26.729860,34.146443,0.146971,20.814231,26.665485,12.769737,31,31,Lahore,Punjab,31.433204,74.341424,1694.990734,210.646812


In [3]:
static["province"].value_counts()

province
Punjab                         34
Khyber Pakhtunkhwa             31
Balochistan                    27
Sindh                          22
Gilgit-Baltistan               10
Islamabad Capital Territory     1
Azad Kashmir                    1
Name: count, dtype: int64

## 2. Data-quality checks

Every check should be 0 except the row count. The expected row count is 126 districts × 312 months (Jan 2000 to Dec 2025).
`era5_days` / `chirps_days` count how many daily images went into each monthly value. A month built from only a few days would be biased.
The plausibility bounds are deliberately generous. Values outside them would point to a processing bug (e.g. Kelvin not converted to Celsius), not a real climate extreme.

In [4]:
P.quality_report(df)

,check,count,note
0,Rows,39312,expected 126 districts x 312 months = 39312
1,Missing values (any column),0,
2,Duplicate district-month rows,0,
3,Months with < 28 daily ERA5 images,0,incomplete month
4,Months with < 28 daily CHIRPS images,0,incomplete month
5,"tmax_c outside [-40, 55]",0,implausible value
6,"tmin_c outside [-50, 40]",0,implausible value
7,"tmean_c outside [-45, 45]",0,implausible value
8,"rain_mm outside [0, 1500]",0,implausible value
9,"era5_rain_mm outside [0, 1500]",0,implausible value


## 3. Missing-value strategy

The extraction returned **no missing values**: ERA5-Land and CHIRPS are gridded, gap-free products, and every district polygon received pixels.
The pipeline still contains a strategy, because a different boundary file or study period could create gaps:

* **Interior gaps:** linear interpolation in time *within the same district* (`P.fill_missing`). Monthly climate changes smoothly, so the neighbouring months are the best estimate.
* **Not used: global mean imputation.** It would put a national average into Gilgit or Jacobabad and destroy both the geography and the seasonal cycle.
* **Gaps at the start or end of a series** cannot be interpolated and would be dropped.

The model notebooks also create *lag* features, which are missing by construction for the first months of each district. Those rows are dropped there (see `04_feature_engineering`).

In [5]:
df, n_filled = P.fill_missing(df, P.CLIMATE_VARS)
print("values filled by interpolation:", n_filled)

values filled by interpolation: 0


## 4. Unit verification

Conversions done in Earth Engine (`gee/extract_data.py`):

| Variable | Source unit | Conversion | Final unit |
|---|---|---|---|
| Temperatures | Kelvin | − 273.15 | °C |
| ERA5 precipitation | metres per day | sum over days × 1000 | mm / month |
| Solar radiation | J m⁻² per day | ÷ 10⁶ | MJ m⁻² day⁻¹ |
| CHIRPS rainfall | mm per day | sum over days | mm / month |
| Soil moisture | m³ m⁻³ | none | m³ m⁻³ |

Spot checks against known climate: Jacobabad (one of the hottest places on Earth) should have the highest June Tmax. Gilgit-Baltistan districts should be below zero in winter. Lahore should get far more July rain than Quetta.

In [6]:
check = df[df["district"].isin(["Jacobabad", "Karachi", "Lahore", "Quetta", "Gilgit", "Islamabad Capital Territory"])]
check.groupby(["district", "month"])[["tmax_c", "tmin_c", "rain_mm"]].mean().unstack("month").round(1)[
    [("tmax_c", 1), ("tmax_c", 6), ("tmin_c", 1), ("rain_mm", 7)]]

tmax_c       tmin_c rain_mm
month                            1     6      1       7
district                                               
Gilgit                       -14.1  11.4  -21.1    34.5
Islamabad Capital Territory   15.7  35.9    3.4   294.1
Jacobabad                     22.2  42.6    9.1    38.0
Karachi                       24.0  34.8   12.7    84.9
Lahore                        18.3  38.9    6.2   194.2
Quetta                         9.4  32.4   -0.7    11.2

In [7]:
# District with the hottest mean June Tmax (2000-2025): should be in Sindh / eastern Balochistan
june = df[df["month"] == 6].groupby(["district", "province"])["tmax_c"].mean().sort_values(ascending=False)
june.head(5).round(1)

district           province   
Nasirabad          Balochistan    43.1
Jafarabad          Balochistan    43.1
Qambar Shahdadkot  Sindh          43.0
Naushehro Feroze   Sindh          42.8
Jhal Magsi         Balochistan    42.7
Name: tmax_c, dtype: float64

## 5. Time columns and seasons

We use Pakistan's four climatological seasons: **Winter (DJF)**, **Pre-monsoon (MAM)**, **Monsoon (JJAS)** and **Post-monsoon (ON)**.
A winter spans two calendar years, so December is assigned to the *next* year's winter (`season_year`). Dec 2000 + Jan/Feb 2001 = winter 2001.

In [8]:
df = P.add_time_columns(df)
df[["date", "year", "month", "season", "season_year"]].drop_duplicates("month").sort_values("month")

,date,year,month,season,season_year
0,2000-01-01,2000,1,Winter (DJF),2000
1,2000-02-01,2000,2,Winter (DJF),2000
2,2000-03-01,2000,3,Pre-monsoon (MAM),2000
3,2000-04-01,2000,4,Pre-monsoon (MAM),2000
4,2000-05-01,2000,5,Pre-monsoon (MAM),2000
5,2000-06-01,2000,6,Monsoon (JJAS),2000
6,2000-07-01,2000,7,Monsoon (JJAS),2000
7,2000-08-01,2000,8,Monsoon (JJAS),2000
8,2000-09-01,2000,9,Monsoon (JJAS),2000
9,2000-10-01,2000,10,Post-monsoon (ON),2000


## 6. Climatology and anomalies

*Feature engineering step 1.*

$$\text{climatology}_{d,m} = \frac{1}{20}\sum_{y=2000}^{2019} x_{d,m,y} \qquad\qquad \text{anomaly}_{d,m,y} = x_{d,m,y} - \text{climatology}_{d,m}$$

* The anomaly removes the **seasonal cycle** and the **fixed geography**. What remains is how unusual a month was *for that place*.
* **Leakage rule:** the climatology uses **training years only (2000–2019)**. If it used all years, the anomalies of training rows would carry information about 2023–2025, and the test score would be optimistic.
* A consequence we check below: training-period anomalies average exactly 0, while validation/test anomalies need not. A positive mean there is itself a sign of recent warming.

In [9]:
df = P.add_anomalies(df)
df["split"] = np.select([df["year"] <= C.TRAIN_YEARS[1], df["year"] <= C.VAL_YEARS[1]], ["train", "validation"], "test")
df.groupby("split")[[c + "_anom" for c in ["tmax_c", "tmin_c", "tmean_c", "rain_mm"]]].mean().round(3).loc[["train", "validation", "test"]]

,tmax_c_anom,tmin_c_anom,tmean_c_anom,rain_mm_anom
split,,,,
train,-0.000,-0.000,-0.000,-0.000
validation,0.016,0.122,0.059,4.066
test,0.309,0.495,0.371,2.390


## 7. Normalisation / standardisation

The brief asks for normalised data. **Where** it happens matters:

* **Ridge regression and SVR** need standardised inputs. Both are distance/penalty based, so a feature measured in hundreds of mm (rainfall) would otherwise dominate one measured in m³/m³ (soil moisture).
* **Random Forest and XGBoost** split on thresholds, so scaling does not change them.
* The scaler (`StandardScaler`: z = (x − μ)/σ) must be **fitted on training rows only** and then applied to validation and test. Fitting it on all rows leaks the test mean and variance into training.

So standardisation is **not** applied to the saved dataset. It is a step inside each model's scikit-learn `Pipeline` (notebook `05_model_training`), which enforces the fit-on-train-only rule automatically. Here we only show what it does:

In [10]:
from sklearn.preprocessing import StandardScaler
cols = ["tmax_c", "rain_mm", "soil_moisture", "elevation_m"]
train = df[df["split"] == "train"]
sc = StandardScaler().fit(train[cols])
pd.DataFrame({"raw mean": train[cols].mean(), "raw std": train[cols].std(),
              "scaled mean": sc.transform(train[cols]).mean(0), "scaled std": sc.transform(train[cols]).std(0)}).round(3)

,raw mean,raw std,scaled mean,scaled std
tmax_c,25.474,12.853,-0.0,1.0
rain_mm,35.680,51.147,0.0,1.0
soil_moisture,0.171,0.112,-0.0,1.0
elevation_m,939.424,1196.171,0.0,1.0


## 8. Save

In [11]:
df = df.drop(columns="split").sort_values(["district_id", "date"]).reset_index(drop=True)
C.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_parquet(C.DATA_PROCESSED / "climate_monthly.parquet", index=False)
print(df.shape)
df.dtypes.to_frame("dtype").T

(39312, 33)


,district_id,year,month,tmax_c,tmin_c,tmean_c,soil_moisture,solar_mj,era5_rain_mm,rain_mm,era5_days,chirps_days,district,province,lat,lon,area_km2,elevation_m,date,season,season_year,tmax_c_clim,tmin_c_clim,tmean_c_clim,rain_mm_clim,soil_moisture_clim,solar_mj_clim,tmax_c_anom,tmin_c_anom,tmean_c_anom,rain_mm_anom,soil_moisture_anom,solar_mj_anom
dtype,str,int64,int64,float64,float64,float64,float64,float64,float64,float64,int64,int64,str,str,float64,float64,float64,float64,datetime64[us],category,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
